In [ ]:
#Instalación de librerías
%pip install alpaca-py scikit-learn pandas numpy matplotlib seaborn joblib pytz python-dotenv 

In [21]:
# Imports generales
# Manejo de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Fechas
from datetime import datetime

# Alpaca — obtención de datos
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

# Scikit-learn — preprocesamiento
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Scikit-learn — modelos
from sklearn.linear_model import LinearRegression, Ridge, BayesianRidge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

# Scikit-learn — métricas
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Guardar modelos
import joblib

# Configuración visual
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 5)

print("!!Librerías cargadas correctamente")

!!Librerías cargadas correctamente


In [22]:
# Credenciales de Alpaca (Paper Trading)

from dotenv import load_dotenv
import os

# Cargar variables del .env
load_dotenv()

API_KEY = os.getenv("API_KEY")
SECRET_TOKEN = os.getenv("SECRET_TOKEN")

# Validación
if not API_KEY or not SECRET_TOKEN:
    raise ValueError("Faltan credenciales en el archivo .env")

client = StockHistoricalDataClient(API_KEY, SECRET_TOKEN)

print("Cliente de Alpaca conectado")

Cliente de Alpaca conectado


In [23]:
#  Descarga de datos históricos de INTC

request_params = StockBarsRequest(
    symbol_or_symbols="INTC",
    timeframe=TimeFrame.Day,          # Barras diarias
    start=datetime(2019, 1, 1),       # Desde enero 2019
    end=datetime(2024, 12, 31),       # Hasta diciembre 2024
    feed="iex"                        # Feed gratuito
)

# Solicitud a la API
bars = client.get_stock_bars(request_params)

# Convertir a DataFrame
df_raw = bars.df.reset_index()

# Vista previa
print(f"Datos descargados: {df_raw.shape[0]} registros, {df_raw.shape[1]} columnas")
print(f"\nColumnas disponibles:\n{df_raw.columns.tolist()}")
print(f"\nPrimeras filas:")
df_raw.head()

Datos descargados: 1115 registros, 9 columnas

Columnas disponibles:
['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap']

Primeras filas:


,symbol,timestamp,open,high,low,close,volume,trade_count,vwap
0,INTC,2020-07-27 04:00:00+00:00,51.03,51.135,49.51,49.545,2381133.0,13266.0,50.125079
1,INTC,2020-07-28 04:00:00+00:00,49.51,50.190,49.14,49.240,1072342.0,6455.0,49.622185
2,INTC,2020-07-29 04:00:00+00:00,49.43,49.470,47.90,48.085,1115663.0,7770.0,48.271563
3,INTC,2020-07-30 04:00:00+00:00,47.81,48.485,47.59,47.990,538909.0,4731.0,47.943802
4,INTC,2020-07-31 04:00:00+00:00,48.12,48.305,47.00,47.720,524091.0,4089.0,47.639135


In [24]:
# Limpieza inicial del DataFrame

# La columna 'symbol' no aporta información (siempre es "INTC")
df_raw = df_raw.drop(columns=["symbol"])

# Aseguramos que timestamp sea de tipo datetime y lo usamos como índice
df_raw["timestamp"] = pd.to_datetime(df_raw["timestamp"])
df_raw = df_raw.set_index("timestamp")

# Verificamos que no haya nulos
print(f"Valores nulos por columna:")
print(df_raw.isnull().sum())
print(f"\nShape del dataset: {df_raw.shape}")
df_raw.head()

Valores nulos por columna:
open           0
high           0
low            0
close          0
volume         0
trade_count    0
vwap           0
dtype: int64

Shape del dataset: (1115, 7)


,open,high,low,close,volume,trade_count,vwap
timestamp,,,,,,,
2020-07-27 04:00:00+00:00,51.03,51.135,49.51,49.545,2381133.0,13266.0,50.125079
2020-07-28 04:00:00+00:00,49.51,50.190,49.14,49.240,1072342.0,6455.0,49.622185
2020-07-29 04:00:00+00:00,49.43,49.470,47.90,48.085,1115663.0,7770.0,48.271563
2020-07-30 04:00:00+00:00,47.81,48.485,47.59,47.990,538909.0,4731.0,47.943802
2020-07-31 04:00:00+00:00,48.12,48.305,47.00,47.720,524091.0,4089.0,47.639135


In [26]:
# Ingeniería de características

df = df_raw.copy()

# 1 Retorno diario
# Variación porcentual del precio de cierre respecto al día anterior
df["daily_return"] = df["close"].pct_change()

# 2 Medias móviles 
# Promedio del precio de cierre en los últimos N días
df["ma_7"]  = df["close"].rolling(window=7).mean()
df["ma_21"] = df["close"].rolling(window=21).mean()

# 3 Volatilidad
# Qué tanto varía el precio en los últimos 7 días
df["volatility_7"] = df["daily_return"].rolling(window=7).std()

# 4 Rango diario
# Diferencia entre el precio más alto y más bajo del día
df["daily_range"] = df["high"] - df["low"]

# 5 Variable objetivo (target)
# El precio de cierre del día SIGUIENTE es lo que queremos predecir
df["target"] = df["close"].shift(-1)

# Eliminar filas con NaN 
df = df.dropna()

print(f"Características creadas correctamente")
print(f"Shape del dataset: {df.shape}")
print(f"\nColumnas del dataset:")
print(df.columns.tolist())
df.head()

Características creadas correctamente
Shape del dataset: (1094, 13)

Columnas del dataset:
['open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap', 'daily_return', 'ma_7', 'ma_21', 'volatility_7', 'daily_range', 'target']


,open,high,low,close,volume,trade_count,vwap,daily_return,ma_7,ma_21,volatility_7,daily_range,target
timestamp,,,,,,,,,,,,,
2020-08-24 04:00:00+00:00,49.350,49.385,48.820,49.135,308299.0,3046.0,49.061107,-0.003044,48.912143,48.714048,0.007865,0.565,49.445
2020-08-25 04:00:00+00:00,49.305,49.860,49.220,49.445,459442.0,3638.0,49.439455,0.006309,48.989286,48.709286,0.007779,0.640,49.515
2020-08-26 04:00:00+00:00,49.355,49.665,49.240,49.515,237884.0,2183.0,49.484368,0.001416,49.075000,48.722381,0.007756,0.425,49.430
2020-08-27 04:00:00+00:00,49.780,49.920,49.195,49.430,281181.0,2319.0,49.583347,-0.001717,49.188571,48.786429,0.007242,0.725,50.405
2020-08-28 04:00:00+00:00,49.505,50.800,49.495,50.405,460013.0,3417.0,50.323914,0.019725,49.480000,48.901429,0.008787,1.305,51.035


In [27]:
# Normalización de los datos

# Separamos las características (X) del target (y)
# El target no se normaliza — queremos predicciones del precio real
X = df.drop(columns=["target"])
y = df["target"]

# Aplicamos StandardScaler a las características
# StandardScaler transforma cada columna para que tenga:
#   - Media = 0
#   - Desviación estándar = 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convertimos de vuelta a DataFrame para mantener los nombres de columnas
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

print("Normalización completada")
print(f"\nEstadísticas post-normalización:")
print(X_scaled.describe().round(2))

Normalización completada

Estadísticas post-normalización:
          open     high      low    close   volume  trade_count     vwap  \
count  1094.00  1094.00  1094.00  1094.00  1094.00      1094.00  1094.00   
mean      0.00     0.00     0.00    -0.00     0.00        -0.00     0.00   
std       1.00     1.00     1.00     1.00     1.00         1.00     1.00   
min      -1.82    -1.84    -1.83    -1.83    -1.19        -1.89    -1.84   
25%      -0.86    -0.86    -0.85    -0.86    -0.64        -0.69    -0.86   
50%       0.01    -0.01    -0.02    -0.01    -0.25        -0.17    -0.01   
75%       0.83     0.82     0.83     0.83     0.28         0.44     0.83   
max       2.31     2.28     2.24     2.32     7.55         7.48     2.28   

       daily_return     ma_7    ma_21  volatility_7  daily_range  
count       1094.00  1094.00  1094.00       1094.00      1094.00  
mean          -0.00    -0.00     0.00         -0.00        -0.00  
std            1.00     1.00     1.00          1.00    

In [ ]:
# Guardar datos procesados

# Guardamos el dataset procesado en CSV para que el equipo lo use
df.to_csv("../data/intc_processed.csv")
X_scaled.to_csv("../data/intc_X_scaled.csv")
y.to_csv("../data/intc_y.csv")

# Guardamos el scaler entrenado para usarlo en predicciones futuras
joblib.dump(scaler, "../models/scaler.pkl")

print("Dataset procesado guardado en /data/intc_processed.csv")
print("X escalado guardado en /data/intc_X_scaled.csv")
print("y guardado en /data/intc_y.csv")
print("Scaler guardado en /models/scaler.pkl")

Dataset procesado guardado en /data/intc_processed.csv
X escalado guardado en /data/intc_X_scaled.csv
y guardado en /data/intc_y.csv
Scaler guardado en /models/scaler.pkl
